<a href="https://jupyterhub.user.eopf.eodc.eu/hub/user-redirect/git-pull?repo=https://github.com/wietzesuijker/community-notebook-competition&branch=submission/cloud-native-vegetation-monitoring&urlpath=lab/tree/community-notebook-competition/submissions/under_the_hood_cloud_native_vegetation_monitoring_with_eopf_zarr/submission.ipynb" target="_blank">
  <button style="background-color:#0072ce; color:white; padding:0.6em 1.2em; font-size:1rem; border:none; border-radius:6px; margin-top:1em;">
    🚀 Launch this notebook in JupyterLab
  </button>
</a>

[Browse the EOPF Zarr Sample Service](https://explorer.eopf.copernicus.eu) — the data source for this notebook.

# Under the Hood: Cloud-Native Vegetation Monitoring with EOPF Zarr

### Introduction

Can we detect spring green-up across an Alpine-to-lowland elevation gradient using only partial cloud-native reads from a multi-terabyte archive?

This notebook answers that question using Sentinel-2 data from the [EOPF Explorer](https://explorer.eopf.copernicus.eu) — ESA's cloud-native access point for Copernicus data. Each analysis step introduces a cloud-native technique along the way:

- Zooming from country scale to individual fields
- Reading only the pixels we need
- Filtering out clouds using co-located quality masks
- Comparing vegetation indices across elevations

All without downloading a single full image.

**Study area**: Friuli-Venezia Giulia, NE Italy (tile T33TVM). From the Julian Alps to the Po Plain — 2,000 m of elevation change where spring arrives weeks apart. In between, the Collio wine hills give us a mid-elevation reference at field-scale detail.

**Data**: [EOPF Zarr Sample Service](https://explorer.eopf.copernicus.eu) — Sentinel-2 L2A, stored in Zarr v3 format. Sentinel-2 captures 13 spectral bands (visible through shortwave infrared) at 10–60 m resolution every 5 days. L2A means the data is atmospherically corrected — surface reflectance, ready for analysis.

### What we will learn

- 🔎 How to zoom from a country-wide overview to field-level detail without downloading the whole image
- 📦 How sharding and consolidated metadata let us fetch just an index instead of hundreds of files
- ⚡ How the data is organized so we can read just the pixels we need — and why that matters for bandwidth and speed
- 🌱 How to detect vegetation change across an elevation gradient using spectral indices and cloud-free pixel selection
- 📊 How to audit exactly what was transferred and verify format consistency

### Prerequisites

This notebook uses the [EOPF Zarr Sample Service STAC API](https://api.explorer.eopf.copernicus.eu/stac) (no authentication required). Key packages:

- [`zarr`](https://zarr.readthedocs.io/) (3.x) — reading remote Zarr stores
- [`fsspec`](https://filesystem-spec.readthedocs.io/) + `aiohttp` — HTTP access to cloud storage
- [`pystac-client`](https://pystac-client.readthedocs.io/) — searching the data catalog
- [`matplotlib`](https://matplotlib.org/) + `numpy` — plotting and computation

<hr>

## 1. Setup

#### Import libraries

In [ ]:
# %pip install -q "zarr>=3.0.7" fsspec aiohttp pystac-client matplotlib numpy

In [ ]:
import json  # parse consolidated metadata
import time  # performance timing
import urllib.request  # probe zarr.json format version
import warnings  # suppress noisy matplotlib warnings
from math import ceil  # sub-chunk count estimates

import matplotlib.pyplot as plt  # publication-quality figures
import numpy as np  # array computation
import zarr  # Zarr v3 array reads with sharding
from matplotlib.colors import BoundaryNorm, ListedColormap  # custom SCL colormap
from matplotlib.gridspec import GridSpec  # multi-panel figure layouts
from matplotlib.patches import ConnectionPatch, Patch, Rectangle  # overlays & legends
from pystac_client import Client  # STAC catalog search
from zarr.storage import FsspecStore

warnings.filterwarnings("ignore", category=UserWarning)

# --- Publication-quality figure style ---
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)

#### Helper functions

These five helpers handle the plumbing so we can focus on the analysis:

- **`timed_read`** — reads a slice from a remote Zarr array, logging bytes and timing for the performance summary in Section 8
- **`compute_index`** — computes any normalized difference index (NDVI, NDRE, etc.) from two bands at any resolution
- **`geo_extent`** — converts the store's spatial metadata into map coordinates for plotting
- **`geo_extent_aoi`** — same, but for a smaller area of interest
- **`geo_to_pixel`** — converts UTM coordinates to pixel positions so we can slice exactly the area we want

In [ ]:
# --- Performance tracking ---
perf_log = []
budget_mb = 0  # running total


def timed_read(array, slices=None, label=""):
    """Read a zarr array slice, log timing and bytes."""
    global budget_mb
    t0 = time.time()
    data = array[:] if slices is None else array[slices]
    elapsed = time.time() - t0
    nbytes = data.nbytes
    budget_mb += nbytes / 1e6
    perf_log.append(
        {
            "label": label,
            "shape": data.shape,
            "bytes": nbytes,
            "seconds": round(elapsed, 2),
        }
    )
    return data


def compute_index(scene, band_a, band_b, aoi, res_group="r10m", label=""):
    """Normalized difference index: (A - B) / (A + B). Clipped to [-1, 1]."""
    grp = scene[f"measurements/reflectance/{res_group}"]
    a = timed_read(grp[band_a], aoi, label=f"{label} {band_a} {res_group}").astype(np.float32)
    b = timed_read(grp[band_b], aoi, label=f"{label} {band_b} {res_group}").astype(np.float32)
    with np.errstate(divide="ignore", invalid="ignore"):
        idx = np.where((a + b) > 0, (a - b) / (a + b), np.nan)
    return np.clip(idx, -1, 1)


def geo_extent(transform, shape, res_factor=1):
    """Compute imshow extent [left, right, bottom, top] from spatial:transform."""
    dx, _, ox, _, dy, oy = transform
    dx *= res_factor
    dy *= res_factor
    nrows, ncols = shape[:2]
    return [ox, ox + ncols * dx, oy + nrows * dy, oy]


def geo_extent_aoi(transform, aoi, res_factor=1):
    """Compute imshow extent for a pixel-coordinate AOI slice."""
    dx, _, ox, _, dy, oy = transform
    dx *= res_factor
    dy *= res_factor
    r0, r1 = aoi[0].start, aoi[0].stop
    c0, c1 = aoi[1].start, aoi[1].stop
    return [ox + c0 * dx, ox + c1 * dx, oy + r1 * dy, oy + r0 * dy]


def geo_to_pixel(transform, easting, northing, res_factor=1):
    """Convert geographic coordinates to pixel row, col."""
    dx, _, ox, _, dy, oy = transform
    dx *= res_factor
    dy *= res_factor
    col = int((easting - ox) / dx)
    row = int((northing - oy) / dy)
    return row, col

<hr>

## 2. STAC Discovery

To measure the elevation gradient, we need three snapshots: a dormant baseline, an early-spring image while the gradient is widening, and a peak-spring reference to show where it ends.

| Scene | Date | Role |
|-------|------|------|
| **Autumn baseline** | November 2025 | Dormant vegetation across all elevations — a common zero before the gradient activates |
| **Early spring** | March 2026 | Green-up underway in the lowlands, still dormant in the Alps — the moment of maximum contrast |
| **Peak spring reference** | May 2025 | Full canopy cover at all elevations — shows where the gradient ends |

Comparing the March–November and May–November differences, zone by zone, reveals how much the green wave has advanced at each elevation.

The [STAC API](https://stacspec.org) (SpatioTemporal Asset Catalog) is the standard way to find these scenes — filter by location, date range, and cloud cover rather than browsing file trees.

Why this tile? Friuli-Venezia Giulia packs an extraordinary elevation range into a single Sentinel-2 scene. Tile T33TVM captures a 110 km north-south transect: alpine peaks above 2,000 m in the Julian Alps, rolling hills at 200 m in the Collio wine country, and the flat Po Plain near sea level. That compressed gradient — sea level to ski altitude in a single scene — makes it a natural laboratory: leaf emergence arrives weeks later at the top than at the bottom, and one tile captures the entire transition.

In [ ]:
STAC_URL = "https://api.explorer.eopf.copernicus.eu/stac"
TILE = "T33TVM"  # Friuli-Venezia Giulia, NE Italy
BBOX = [13.7, 46.0, 15.1, 47.0]

client = Client.open(STAC_URL)

# --- Latest scene: most recent clear pass (R122 covers the full tile) ---
latest_results = client.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2026-03-01/2026-03-15",
    query={"eo:cloud_cover": {"lt": 5}},
    max_items=20,
    sortby=["-properties.datetime"],
)
latest_item = next(item for item in latest_results.items() if TILE in item.id and "R122" in item.id)

# --- Baseline scene: late autumn 2025, dormant vegetation ---
baseline_results = client.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2025-11-01/2025-11-30",
    query={"eo:cloud_cover": {"lt": 5}},
    max_items=20,
)
baseline_item = next(
    item for item in baseline_results.items() if TILE in item.id and "R122" in item.id
)

# --- Peak spring scene: May 2025, when elevation gradient is visible ---
peak_results = client.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2025-05-01/2025-06-15",
    query={"eo:cloud_cover": {"lt": 15}},
    max_items=20,
)
peak_item = next(item for item in peak_results.items() if TILE in item.id and "R122" in item.id)

for label, item in [
    ("SPRING (latest)", latest_item),
    ("AUTUMN (baseline)", baseline_item),
    ("PEAK SPRING (2025)", peak_item),
]:
    refl = item.assets["reflectance"]
    print(f"{label}")
    print(f"  {item.id}")
    print(
        f"  {item.datetime.strftime('%Y-%m-%d %H:%M UTC')}  |  cloud: {item.properties['eo:cloud_cover']:.1f}%"
    )
    print(f"  media type: {refl.media_type}")
    print()

### When metadata and data disagree

The STAC metadata declares `version=2`. Before querying the data, let’s verify it.

In evolving data systems, metadata lags behind the data it describes. A version mismatch between catalog and store will trip up any client that trusts the media type without checking. This specific mismatch has been [fixed in the EOPF data pipeline](https://github.com/EOPF-Explorer/data-pipeline/pull/94), though existing STAC items haven’t been re-registered yet. The probing technique below works regardless.

> 💡 **The takeaway: always probe the data directly rather than trusting catalog metadata alone.** We’ll see this pattern pay off again in Section 9.

In [ ]:
# Probe the actual format by fetching the root zarr.json
zarr_root_url = latest_item.assets["reflectance"].href.rsplit("/measurements/", 1)[0]

resp = urllib.request.urlopen(f"{zarr_root_url}/zarr.json", timeout=15)
root_meta = json.loads(resp.read())

stac_version = latest_item.assets["reflectance"].media_type
actual_version = root_meta["zarr_format"]

print(f"STAC declares:  {stac_version}")
print(f"zarr.json says: zarr_format = {actual_version}")
print("\nMismatch: the data is Zarr v3, but STAC advertises v2.")

### The Official Toolkit Path: `xarray-eopf`

Before diving into the raw store internals, it's worth knowing the **official EOPF toolkit entry point**: the `xarray-eopf` package registers an `"eopf-zarr"` xarray backend that opens any EOPF Zarr store as a tidy Dataset in one line:

```python
import xarray as xr
import xarray_eopf  # pip install xarray-eopf; registers engine="eopf-zarr"

ds = xr.open_dataset(scene_url, engine="eopf-zarr")
print(ds)  # → xarray Dataset with named variables, CRS, and CF metadata
```

We recommend this path for production workflows — it handles CRS, variable naming, and CF (Climate and Forecast) metadata automatically.

**Why we go deeper here.** `xarray-eopf` currently targets Zarr v2 stores, while the EOPF sample service delivers Zarr v3 data (verified in the next cell). This notebook uses zarr 3.x directly to inspect sharding, codec chains, and multi-resolution groups — the internals that any high-level API must navigate. Understanding what's underneath makes you a better user of any EOPF-aware tool: you'll know what `engine="eopf-zarr"` does behind the scenes, what to check when something fails, and how to read any zarr.json yourself.

Both paths read the same EOPF Zarr archive. One gives you a polished Dataset. The other shows you every byte.

## 3. Inside the Store

Before reading any pixels, we need to know what's available — which bands, which resolutions, what coordinate system, how the data is chunked.

Zarr v3 makes this cheap: a single file called `zarr.json` at the root acts as a **table of contents** for the entire dataset. One HTTP request, and we know everything about the store's structure. Compare that to Zarr v2, where the same information was scattered across individual metadata files — over 210 separate HTTP requests for a store this size.

Inside the store, EOPF organizes data like a well-labeled filing cabinet. The top-level drawers separate science bands (`measurements/`), quality masks (`conditions/`), geometry, and metadata. Within `measurements/`, data is grouped by resolution: `r10m/b04`, `r20m/b05`, and so on. If we only need 10m bands, we never touch the 20m or 60m data.

In [ ]:
consolidated = root_meta.get("consolidated_metadata", {}).get("metadata", {})
groups = [
    k for k, v in consolidated.items() if isinstance(v, dict) and v.get("node_type") == "group"
]
arrays = [
    k for k, v in consolidated.items() if isinstance(v, dict) and v.get("node_type") == "array"
]

print(f"Consolidated metadata: {len(groups)} groups, {len(arrays)} arrays")
print("(All discovered from a single zarr.json request)\n")

# Show the data model — top-level groups + reflectance detail
print("Data model (top-level + reflectance detail):")
for g in sorted(groups):
    depth = g.count("/")
    if depth == 0:
        print(f"  {g}/")
    elif depth == 1 and ("measurements" in g or "conditions" in g):
        print(f"    {g}/")
    elif "reflectance" in g:
        print(f"      {g}/")

# Extract the spatial reference we'll use throughout
r10m_attrs = consolidated.get("measurements/reflectance/r10m", {}).get("attributes", {})
TRANSFORM = r10m_attrs["spatial:transform"]  # [dx, rot_x, origin_x, rot_y, dy, origin_y]
CRS = r10m_attrs["proj:code"]
BBOX_UTM = r10m_attrs["spatial:bbox"]

print(f"\nCRS: {CRS}")
print(f"Spatial extent: {BBOX_UTM}")

### Sharding: the key to fewer HTTP requests

In Zarr v2, every chunk of data was a separate file on the server. A single 10m band could mean over a hundred files, each requiring its own HTTP request.

Zarr v3 introduced **sharding**: packing many chunks into a single large file with an index at the end. To read a specific area, the client first fetches this small index (one request), which tells it exactly where each chunk sits inside the file. Then it fetches only the chunks that overlap the area of interest — skipping everything else.

> 💡 **One index lookup replaces a hundred individual file requests — that's the difference sharding makes.**

In [ ]:
# Inspect the codec chain for a 10m band
b04_meta = consolidated["measurements/reflectance/r10m/b04"]
shard_shape = b04_meta["chunk_grid"]["configuration"]["chunk_shape"]
sharding_codec = b04_meta["codecs"][0]
sub_shape = sharding_codec["configuration"]["chunk_shape"]
sub_codecs = [c["name"] for c in sharding_codec["configuration"]["codecs"]]

print(f"Array shape:  {b04_meta['shape']}")
print(f"Shard shape:  {shard_shape}")
print(f"Sub-chunks:   {sub_shape}")
print(f"Codec chain:  sharding_indexed \u2192 {sub_codecs}")
print(f"Index at:     {sharding_codec['configuration'].get('index_location', 'end')}")

# Quantify the sharding advantage
n_subchunks = int(np.prod([ceil(10980 / s) for s in sub_shape]))
n_shards = int(
    np.prod([ceil(s1 / s2) for s1, s2 in zip(b04_meta["shape"], shard_shape, strict=False)])
)
subchunks_per_shard = int(
    np.prod([s1 // s2 for s1, s2 in zip(shard_shape, sub_shape, strict=False)])
)
n_side = int(ceil(10980 / sub_shape[0]))

print(f"\nWithout sharding: {n_subchunks} files = {n_subchunks} HTTP requests for the full band")
print(f"With sharding:    {n_shards} shard file, {subchunks_per_shard} sub-chunks packed inside")

# --- Shard layout visualization ---
fig, ax = plt.subplots(figsize=(7, 7))

# Draw the 12x12 sub-chunk grid
for r in range(n_side):
    for c in range(n_side):
        color = "#e8e8e8"
        ax.add_patch(
            Rectangle((c, n_side - 1 - r), 1, 1, facecolor=color, edgecolor="#999", linewidth=0.5)
        )

# Highlight the sub-chunks touched by a 5x5 km AOI (rows 9-10, cols 0-1)
aoi_chunks = [(r, c) for r in range(9, 11) for c in range(0, 2)]
for r, c in aoi_chunks:
    ax.add_patch(
        Rectangle(
            (c, n_side - 1 - r),
            1,
            1,
            facecolor="#2ecc71",
            edgecolor="#006400",
            linewidth=1.5,
            alpha=0.7,
        )
    )

# Index marker at bottom
ax.add_patch(
    Rectangle((0, -0.6), n_side, 0.4, facecolor="#3498db", edgecolor="#1a5276", linewidth=1)
)
ax.text(
    n_side / 2,
    -0.4,
    "BINARY INDEX",
    ha="center",
    va="center",
    fontsize=9,
    fontweight="bold",
    color="white",
)

ax.set_xlim(-0.2, n_side + 0.2)
ax.set_ylim(-1, n_side + 0.2)
ax.set_aspect("equal")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
ax.set_title(
    f"Shard layout: {n_side}\u00d7{n_side} = {subchunks_per_shard} sub-chunks in 1 file\n"
    f"Green = sub-chunks touched by a 5\u00d75 km AOI read ({len(aoi_chunks)} of {subchunks_per_shard})"
)

# Custom legend

ax.legend(
    handles=[
        Patch(facecolor="#e8e8e8", edgecolor="#999", label="Sub-chunk (not read)"),
        Patch(facecolor="#2ecc71", edgecolor="#006400", label="Sub-chunk (AOI read)"),
        Patch(facecolor="#3498db", edgecolor="#1a5276", label="Binary index (1 request)"),
    ],
    loc="upper right",
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(
    f"Reading a 5\u00d75 km AOI: 1 index request + {len(aoi_chunks)} data requests = {len(aoi_chunks) + 1} HTTP requests total"
)


def estimate_read_mb(shape, aoi, sub_chunk_shape, dtype_bytes=4):
    """Estimate bytes transferred for a partial read (upper bound: full sub-chunks touched)."""
    r0, r1 = aoi[0].start, aoi[0].stop
    c0, c1 = aoi[1].start, aoi[1].stop
    chunks_row = ceil((r1 - r0) / sub_chunk_shape[0])
    chunks_col = ceil((c1 - c0) / sub_chunk_shape[1])
    chunk_bytes = sub_chunk_shape[0] * sub_chunk_shape[1] * dtype_bytes
    return chunks_row * chunks_col * chunk_bytes / 1e6


# Preview: how much will a 500x500 AOI cost?
test_aoi = (slice(4000, 4500), slice(3000, 3500))
est = estimate_read_mb(b04_meta["shape"], test_aoi, sub_shape)
print(
    f"Cost estimate for 500\u00d7500 AOI read: ~{est:.1f} MB (sub-chunks touched, before compression)"
)

## 4. Progressive Zoom: From Continent to Field

Opening a Zarr store doesn't download data — it creates a lightweight reference. Actual pixels are fetched on demand when you slice the array, so browsing the store structure is essentially free.

Before computing indices on a tiny AOI, let's see what the full scene looks like. This is where multi-resolution groups pay off: instead of downloading the full 10980×10980 tile (~240 MB per band), we start at the coarsest overview and zoom in progressively. Each level reads from a different group in the same store — no separate files, no server-side processing.

The coarser resolution groups (r720m, r360m, r120m) are **overview pyramids** added by the EOPF Explorer on top of the native Sentinel-2 resolutions. They exist so clients can preview scenes cheaply before committing to a full-resolution read.

In [ ]:
def open_scene(item):
    """Open an EOPF Zarr scene from a STAC item."""
    root_url = item.assets["reflectance"].href.rsplit("/measurements/", 1)[0]
    store = FsspecStore.from_url(root_url)
    return zarr.open_group(store, mode="r")


spring = open_scene(latest_item)
autumn = open_scene(baseline_item)
peak = open_scene(peak_item)

# Show available resolution levels
refl = spring["measurements/reflectance"]
print("Resolution groups in this store:")
for name, node in refl.members():
    if hasattr(node, "members"):
        bands = [n for n, _ in node.members() if n.startswith("b")]
        shape_info = ""
        for n, arr in node.members():
            if n.startswith("b") and hasattr(arr, "shape"):
                shape_info = f"{arr.shape[0]}x{arr.shape[1]}"
                break
        print(f"  {name}: {shape_info} ({len(bands)} bands)")

### Three levels, one store

We'll load the spring scene at three zoom levels: r720m (thumbnail), r360m (regional context), and r10m (a 5 km area of interest over the Collio wine hills). Each read comes from a different resolution group in the same Zarr store.

In the overview, look for the snow line separating white Alps from green lowlands — that's a visual preview of the elevation-dependent green-up we'll quantify in Section 7. In the 10m detail, individual field boundaries and vineyard rows emerge — the scale at which field-scale vegetation monitoring operates.

All zoom levels share the same coordinate grid, so we don't need any reprojection code to line them up.

In [ ]:
def load_rgb(scene, res_group, aoi=None, label_prefix=""):
    """Load true-color RGB (B04=R, B03=G, B02=B) from a resolution group."""
    grp = scene[f"measurements/reflectance/{res_group}"]
    r = timed_read(grp["b04"], aoi, label=f"{label_prefix} {res_group} B04")
    g = timed_read(grp["b03"], aoi, label=f"{label_prefix} {res_group} B03")
    b = timed_read(grp["b02"], aoi, label=f"{label_prefix} {res_group} B02")
    rgb = np.stack([r, g, b], axis=-1).astype(np.float32)
    rgb = np.clip(rgb * 3.5, 0, 1)
    rgb[rgb.sum(axis=-1) == 0] = np.nan  # nodata -> transparent
    return rgb


# --- Define the AOI geographically, then convert to pixels ---
# Collio wine region, hills between Gorizia and Udine
# These UTM coords cover 5x5 km of mixed agriculture/vineyards
AOI_EAST, AOI_WEST = 405000, 410000  # easting (meters)
AOI_NORTH, AOI_SOUTH = 5110000, 5105000  # northing (meters)

r0, c0 = geo_to_pixel(TRANSFORM, AOI_EAST, AOI_NORTH)
r1, c1 = geo_to_pixel(TRANSFORM, AOI_WEST, AOI_SOUTH)
AOI_10m = (slice(r0, r1), slice(c0, c1))
# 10m pixel indices halved = 20m pixel indices (same geographic extent)
AOI_20m = (slice(r0 // 2, r1 // 2), slice(c0 // 2, c1 // 2))

print(f"AOI: {AOI_EAST}-{AOI_WEST} E, {AOI_SOUTH}-{AOI_NORTH} N ({CRS})")
print(
    f"Pixel coords at 10m: rows {r0}:{r1}, cols {c0}:{c1} = {r1 - r0}x{c1 - c0} pixels = {(r1 - r0) * 10 / 1000:.0f}x{(c1 - c0) * 10 / 1000:.0f} km"
)

# --- Load three zoom levels ---
rgb_720m = load_rgb(spring, "r720m", label_prefix="zoom")
rgb_360m = load_rgb(spring, "r360m", label_prefix="zoom")
rgb_10m = load_rgb(spring, "r10m", AOI_10m, label_prefix="zoom")

# Percentile stretch for the 10m AOI — reveals vineyard/field textures
for ch in range(3):
    band = rgb_10m[:, :, ch]
    valid = band[np.isfinite(band) & (band > 0)]
    lo, hi = np.percentile(valid, [2, 98])
    rgb_10m[:, :, ch] = np.clip((band - lo) / (hi - lo), 0, 1)

# --- Progressive zoom composite ---
ext_full = geo_extent(TRANSFORM, rgb_360m.shape, res_factor=36)
ext_720 = geo_extent(TRANSFORM, rgb_720m.shape, res_factor=72)
ext_aoi = geo_extent_aoi(TRANSFORM, AOI_10m)

# Regional extent for the middle panel (~60x50 km around the AOI)
REG_LEFT, REG_RIGHT = 402000, 442000
REG_BOT, REG_TOP = 5092000, 5132000


# Height ratios match geographic aspect ratios so aspect="equal" fills each panel
# Panel 0: full tile ~109.8 x 109.8 km -> h/w = 1.0
# Panel 1: regional crop 60 x 50 km -> h/w = 50/60 = 0.833
# Panel 2: AOI 5 x 5 km -> h/w = 1.0
hw_ratios = [1.0, (REG_TOP - REG_BOT) / (REG_RIGHT - REG_LEFT), 1.0]
FIG_W = 10
fig = plt.figure(figsize=(FIG_W, FIG_W * sum(hw_ratios) + 2))
gs = GridSpec(3, 1, figure=fig, height_ratios=hw_ratios, hspace=0.08)
axes = [fig.add_subplot(gs[i]) for i in range(3)]

# --- Panel 0: r720m overview (full tile) ---
axes[0].imshow(rgb_720m, extent=ext_720, aspect="equal")
axes[0].set_title("r720m — tile overview")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
# Geographic labels
mid_x = (ext_720[0] + ext_720[1]) / 2
axes[0].text(
    mid_x,
    ext_720[3] - (ext_720[3] - ext_720[2]) * 0.08,
    "Julian Alps",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="white",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.5),
)
axes[0].text(
    mid_x,
    ext_720[2] + (ext_720[3] - ext_720[2]) * 0.08,
    "Po Plain",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="white",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.5),
)
# Regional box (dashed red)
reg_rect = Rectangle(
    (REG_LEFT, REG_BOT),
    REG_RIGHT - REG_LEFT,
    REG_TOP - REG_BOT,
    linewidth=2,
    edgecolor="red",
    facecolor="none",
    linestyle="--",
)
axes[0].add_patch(reg_rect)

# --- Panel 1: r360m regional (zoomed to ~60x50 km) ---
axes[1].imshow(rgb_360m, extent=ext_full, aspect="equal")
axes[1].set_xlim(REG_LEFT, REG_RIGHT)
axes[1].set_ylim(REG_BOT, REG_TOP)
axes[1].set_title("r360m — regional context")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")
# AOI box (solid red)
aoi_rect = Rectangle(
    (ext_aoi[0], ext_aoi[2]),
    ext_aoi[1] - ext_aoi[0],
    ext_aoi[3] - ext_aoi[2],
    linewidth=2,
    edgecolor="red",
    facecolor="none",
)
axes[1].add_patch(aoi_rect)
axes[1].annotate(
    "Collio Hills\n(5 km AOI)",
    xy=(ext_aoi[1], (ext_aoi[2] + ext_aoi[3]) / 2),
    xytext=(ext_aoi[1] + 8000, (ext_aoi[2] + ext_aoi[3]) / 2),
    fontsize=10,
    fontweight="bold",
    color="red",
    arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="red", alpha=0.8),
)

# --- Panel 2: r10m AOI detail ---
axes[2].imshow(rgb_10m, extent=ext_aoi, aspect="equal")
axes[2].set_title("r10m — field boundaries visible")
axes[2].set_xlabel("Easting (m)")
axes[2].set_ylabel("Northing (m)")

# --- Connectors between panels (vertical: bottom of panel N → top of panel N+1) ---
for xy_a, xy_b in [
    ((REG_LEFT, REG_BOT), (REG_LEFT, REG_TOP)),
    ((REG_RIGHT, REG_BOT), (REG_RIGHT, REG_TOP)),
]:
    con = ConnectionPatch(
        xyA=xy_a,
        xyB=xy_b,
        coordsA="data",
        coordsB="data",
        axesA=axes[0],
        axesB=axes[1],
        color="red",
        lw=1,
        linestyle="--",
        alpha=0.6,
    )
    fig.add_artist(con)

for xy_a, xy_b in [
    ((ext_aoi[0], ext_aoi[2]), (ext_aoi[0], ext_aoi[3])),
    ((ext_aoi[1], ext_aoi[2]), (ext_aoi[1], ext_aoi[3])),
]:
    con = ConnectionPatch(
        xyA=xy_a,
        xyB=xy_b,
        coordsA="data",
        coordsB="data",
        axesA=axes[1],
        axesB=axes[2],
        color="red",
        lw=1,
        linestyle="--",
        alpha=0.6,
    )
    fig.add_artist(con)

fig.suptitle(
    f"Progressive Zoom: {latest_item.datetime.strftime('%Y-%m-%d')}  —  same store, three resolution groups",
    fontsize=14,
    y=0.995,
)
plt.show()

From 720m overview to 10m field detail in three zoom levels, all from the same store. The snow-capped Julian Alps in the north contrast sharply with the green lowlands in the south — a visible preview of the elevation-dependent green-up we'll quantify in Section 7.

## 5. Trust the Pixels: Cloud Masking with SCL

With the full scene and field-scale detail in view, the next challenge is pixel quality. Not every pixel is trustworthy. Before computing vegetation indices, we need to filter out clouds, shadows, and snow. The Scene Classification Layer (SCL) stores a per-pixel label right alongside the reflectance data in the same store — no separate download, no grid alignment needed.

We keep pixels labeled as vegetation (4), bare soil (5), or unclassified (7). Clouds, cloud shadows, water, snow, and saturated pixels get filtered out.

Why keep “unclassified” (class 7)? Watch out for this: The SCL classifier assigns this label when it can't confidently categorize a pixel — which happens often under the low sun angles of autumn scenes. These are perfectly valid land surface pixels; discard them, and you lose a significant fraction of your usable data in winter acquisitions.

In [ ]:
SCL_LABELS = {
    0: "No data",
    1: "Saturated",
    2: "Dark/shadow",
    3: "Cloud shadow",
    4: "Vegetation",
    5: "Bare soil",
    6: "Water",
    7: "Unclassified",
    8: "Cloud medium",
    9: "Cloud high",
    10: "Cirrus",
    11: "Snow/ice",
}
SCL_COLORS = {
    0: "#000000",
    1: "#ff0000",
    2: "#404040",
    3: "#8B4513",
    4: "#00aa00",
    5: "#c8b464",
    6: "#0000ff",
    7: "#808080",
    8: "#c0c0c0",
    9: "#ffffff",
    10: "#add8e6",
    11: "#00ffff",
}

# Vegetation analysis: vegetation, bare soil, and unclassified
# Class 7 ("unclassified") captures valid land surface that the SCL classifier
# couldn't confidently assign — common in autumn low-sun conditions
VALID_CLASSES = {4, 5, 7}

# Load full-tile SCL at r20m for all three dates
spring_scl_full = timed_read(
    spring["conditions/mask/l2a_classification/r20m/scl"], label="spring SCL r20m"
)
autumn_scl_full = timed_read(
    autumn["conditions/mask/l2a_classification/r20m/scl"], label="autumn SCL r20m"
)
peak_scl_full = timed_read(
    peak["conditions/mask/l2a_classification/r20m/scl"], label="peak SCL r20m"
)

# --- SCL spatial map (top) + class distributions (bottom) ---

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 3, figure=fig, height_ratios=[2, 1], hspace=0.35, wspace=0.45)

# Top: spatial SCL map spanning full width
ax_map = fig.add_subplot(gs[0, :])
cmap_scl = ListedColormap([SCL_COLORS[i] for i in range(12)])
norm_scl = BoundaryNorm(range(13), cmap_scl.N)

ext_20m = geo_extent(TRANSFORM, spring_scl_full.shape, res_factor=2)
ax_map.imshow(
    spring_scl_full,
    cmap=cmap_scl,
    norm=norm_scl,
    extent=ext_20m,
    aspect="equal",
    interpolation="nearest",
)
ax_map.set_title(f"Spring SCL (spatial) — {latest_item.datetime.strftime('%Y-%m-%d')}", fontsize=13)
ax_map.set_xlabel("Easting (m)")
ax_map.set_ylabel("Northing (m)")
# Add AOI box
aoi_ext = geo_extent_aoi(TRANSFORM, AOI_10m)
rect = Rectangle(
    (aoi_ext[0], aoi_ext[2]),
    aoi_ext[1] - aoi_ext[0],
    aoi_ext[3] - aoi_ext[2],
    linewidth=2,
    edgecolor="red",
    facecolor="none",
)
ax_map.add_patch(rect)

# Bottom row: bar charts for all three dates
for idx, (scl, title) in enumerate(
    [
        (autumn_scl_full, f"Autumn {baseline_item.datetime.strftime('%Y-%m')}"),
        (spring_scl_full, f"Spring {latest_item.datetime.strftime('%Y-%m')}"),
        (peak_scl_full, f"Peak {peak_item.datetime.strftime('%Y-%m')}"),
    ]
):
    ax = fig.add_subplot(gs[1, idx])
    unique, counts = np.unique(scl, return_counts=True)
    total = counts.sum()
    colors = [SCL_COLORS.get(u, "#999") for u in unique]
    labels = [SCL_LABELS.get(u, str(u)) for u in unique]
    bars = ax.barh(labels, counts / total * 100, color=colors, edgecolor="black", linewidth=0.5)
    # Highlight valid classes
    for bar, u in zip(bars, unique, strict=False):
        if u in VALID_CLASSES:
            bar.set_edgecolor("red")
            bar.set_linewidth(2)
    ax.set_xlabel("% of pixels")
    ax.set_title(f"{title} SCL classes\n(red border = valid for analysis)")
    ax.set_xlim(0, 100)
    ax.tick_params(axis="y", labelsize=9)

plt.show()

spring_valid = np.isin(spring_scl_full, list(VALID_CLASSES))
autumn_valid = np.isin(autumn_scl_full, list(VALID_CLASSES))
peak_valid = np.isin(peak_scl_full, list(VALID_CLASSES))
print(f"Spring valid pixels: {spring_valid.mean() * 100:.1f}%")
print(f"Autumn valid pixels: {autumn_valid.mean() * 100:.1f}%")
print(f"Peak spring valid pixels: {peak_valid.mean() * 100:.1f}%")

The May 2025 scene has higher cloud cover (12.8%) than the other two — the valid-pixel percentages show how much usable area remains after SCL filtering.

## 6. Where is Spring? NDVI and Red-Edge Change Detection

Vegetation indices measure how green a surface is by comparing how leaves interact with different wavelengths of light. The key insight: healthy leaves absorb red light for photosynthesis and strongly reflect near-infrared (NIR). The bigger the gap between these two, the greener the surface.

**NDVI** = (NIR − Red) / (NIR + Red), using bands B08 (842 nm) and B04 (665 nm) at **10m** resolution. Values range from −1 to +1: water and bare rock fall below 0.1, sparse vegetation around 0.2–0.4, dense healthy canopy above 0.6 (Rouse et al., 1974). Dividing by the sum normalizes for lighting differences between dates, making it reliable for multi-temporal comparison.

But NDVI has a ceiling: once a canopy is dense enough, adding more leaves barely changes the score. **NDRE** = (B8A − B05) / (B8A + B05) uses the red-edge band at 20m resolution. The red-edge spectrum (705–783 nm) is sensitive to early chlorophyll changes that saturate NDVI — making NDRE better at catching the first subtle signs of spring (Gitelson & Merzlyak, 1994). For vineyards like the Collio DOC — visible at the center of the AOI — NDRE at 20m resolves individual parcels, making canopy management decisions possible at a scale that 250–500m traditional products cannot reach.

Both indices come from the same store, just different resolution groups. The grids align by design — no reprojection needed.

At the AOI scale, the March green-up is clear. But does this change vary with elevation? In In Section 7, we test whether the signal varies with elevation — and why one date isn't enough.

In [ ]:
# --- NDVI at 10m ---
spring_ndvi = compute_index(spring, "b08", "b04", AOI_10m, "r10m", "spring")
autumn_ndvi = compute_index(autumn, "b08", "b04", AOI_10m, "r10m", "autumn")

# --- NDRE at 20m ---
spring_ndre = compute_index(spring, "b8a", "b05", AOI_20m, "r20m", "spring")
autumn_ndre = compute_index(autumn, "b8a", "b05", AOI_20m, "r20m", "autumn")

# --- SCL masks ---
spring_scl_aoi = spring_scl_full[AOI_20m]
autumn_scl_aoi = autumn_scl_full[AOI_20m]
mask_20m = np.isin(spring_scl_aoi, list(VALID_CLASSES)) & np.isin(
    autumn_scl_aoi, list(VALID_CLASSES)
)
mask_10m = np.repeat(np.repeat(mask_20m, 2, axis=0), 2, axis=1)

ndvi_diff = np.where(mask_10m, spring_ndvi - autumn_ndvi, np.nan)
ndre_diff = np.where(mask_20m, spring_ndre - autumn_ndre, np.nan)

# --- Four-panel visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ext_10m = geo_extent_aoi(TRANSFORM, AOI_10m)
ext_20m_aoi = geo_extent_aoi(TRANSFORM, AOI_10m)  # same geographic extent

for ax, data, title, cmap, vmin, vmax, ext in [
    (
        axes[0, 0],
        autumn_ndvi,
        f"NDVI Autumn ({baseline_item.datetime.strftime('%Y-%m-%d')})",
        "YlGn",
        -0.1,
        0.8,
        ext_10m,
    ),
    (
        axes[0, 1],
        spring_ndvi,
        f"NDVI Spring ({latest_item.datetime.strftime('%Y-%m-%d')})",
        "YlGn",
        -0.1,
        0.8,
        ext_10m,
    ),
    (axes[1, 0], ndvi_diff, "NDVI Change (10m)", "RdYlGn", -0.4, 0.4, ext_10m),
    (axes[1, 1], ndre_diff, "NDRE Change (20m) — Red-edge", "RdYlGn", -0.4, 0.4, ext_20m_aoi),
]:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, extent=ext, aspect="equal")
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if "Change" in title or "NDRE" in title:
        cb.ax.text(
            0.5, 0.02, "Browning", transform=cb.ax.transAxes,
            ha="center", va="bottom", fontsize=7, color="white", fontweight="bold",
        )
        cb.ax.text(
            0.5, 0.98, "Greening", transform=cb.ax.transAxes,
            ha="center", va="top", fontsize=7, color="white", fontweight="bold",
        )

# Add location label to first panel
axes[0, 0].text(
    0.03,
    0.97,
    "Collio Hills AOI\n5 x 5 km",
    transform=axes[0, 0].transAxes,
    ha="left",
    va="top",
    fontsize=9,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray", alpha=0.8),
)

valid_pixels = np.nansum(~np.isnan(ndvi_diff))
greenup = np.nansum(ndvi_diff > 0.1) / valid_pixels * 100
stressed = np.nansum(ndvi_diff < -0.1) / valid_pixels * 100

fig.suptitle("Spring Green-Up: NDVI vs Red-Edge Change Detection", fontsize=14, y=1.01)
fig.text(
    0.5,
    -0.01,
    f"NDVI mean change: {np.nanmean(ndvi_diff):+.3f}  |  NDRE mean change: {np.nanmean(ndre_diff):+.3f}\n"
    f"Green-up (NDVI > +0.1): {greenup:.1f}%  |  Declining (< -0.1): {stressed:.1f}%  |  Stable: {100 - greenup - stressed:.1f}%\n"
    f"Compare bottom panels: NDRE reveals early chlorophyll changes where NDVI shows 'stable'.",
    ha="center",
    fontsize=10,
    style="italic",
    color="#444",
)
plt.tight_layout()
plt.show()

## 7. Elevation Gradient: Alps to Po Plain

Spring doesn't arrive everywhere at once. Temperature drops roughly 6.5 °C per 1,000 m (the environmental lapse rate). Across our tile’s 2,000 m range, the Po Plain at ~30 m runs ~10 °C warmer than the Julian Alps at ~1,500 m. Plants track accumulated warmth (growing degree days) to trigger leaf-out, which means lowlands green up weeks before mountain meadows (Menzel et al., 2006).

Can we see this in the data? Not yet — in early March, the gradient hasn't had time to develop. All elevations show similar modest greening. To reveal what the full gradient looks like, we add a **peak-spring reference**: May 2025, when lowlands are fully green and alpine meadows are just emerging from snow.

We compare three dates at three elevation zones, using targeted partial reads and reusing the SCL masks from Section 5.

In [ ]:
# Three elevation zones spanning the full tile's 2,000 m gradient.
# We compare three dates: Nov 2025 (dormant), Mar 2026 (early spring), May 2025 (peak spring).
zones = {
    "Po Plain\n(~30 m)": {"easting": (405000, 409000), "northing": (5099000, 5095000)},
    "Collio Hills\n(~200 m)": {
        "easting": (405000, 409000),
        "northing": (5110000, 5106000),
    },
    "Julian Alps\n(~1,500 m)": {
        "easting": (415000, 419000),
        "northing": (5180000, 5176000),
    },
}

zone_colors = ["#2ecc71", "#f39c12", "#3498db"]
mar_stats = []
may_stats = []
diff_mar_maps = []
diff_may_maps = []
zone_extents = []


def upscale_mask(mask_20m, target_shape):
    """Upscale a 20m boolean mask to 10m by repeating pixels."""
    m = np.repeat(np.repeat(mask_20m, 2, axis=0), 2, axis=1)
    return m[: target_shape[0], : target_shape[1]]


for col, (zone_name, coords) in enumerate(zones.items()):
    r0, c0 = geo_to_pixel(TRANSFORM, coords["easting"][0], coords["northing"][0])
    r1, c1 = geo_to_pixel(TRANSFORM, coords["easting"][1], coords["northing"][1])
    zone_aoi = (slice(r0, r1), slice(c0, c1))

    # NDVI for all three dates
    s_ndvi = compute_index(spring, "b08", "b04", zone_aoi, "r10m", f"elev-spring-{col}")
    a_ndvi = compute_index(autumn, "b08", "b04", zone_aoi, "r10m", f"elev-autumn-{col}")
    p_ndvi = compute_index(peak, "b08", "b04", zone_aoi, "r10m", f"elev-peak-{col}")

    # SCL masks (intersection of valid pixels across date pairs)
    z_scl = (slice(r0 // 2, r1 // 2), slice(c0 // 2, c1 // 2))
    s_scl = spring_scl_full[z_scl]
    a_scl = autumn_scl_full[z_scl]
    p_scl = peak_scl_full[z_scl]
    valid_s = np.isin(s_scl, list(VALID_CLASSES))
    valid_a = np.isin(a_scl, list(VALID_CLASSES))
    valid_p = np.isin(p_scl, list(VALID_CLASSES))

    mask_mar = upscale_mask(valid_s & valid_a, s_ndvi.shape)
    mask_may = upscale_mask(valid_p & valid_a, p_ndvi.shape)

    diff_mar = np.where(
        mask_mar,
        s_ndvi[: mask_mar.shape[0], : mask_mar.shape[1]]
        - a_ndvi[: mask_mar.shape[0], : mask_mar.shape[1]],
        np.nan,
    )
    diff_may = np.where(
        mask_may,
        p_ndvi[: mask_may.shape[0], : mask_may.shape[1]]
        - a_ndvi[: mask_may.shape[0], : mask_may.shape[1]],
        np.nan,
    )

    ext = geo_extent_aoi(TRANSFORM, zone_aoi)
    mar_change = np.nanmean(diff_mar)
    may_change = np.nanmean(diff_may)
    mar_valid = mask_mar.sum() / mask_mar.size * 100
    may_valid = mask_may.sum() / mask_may.size * 100
    mar_stats.append((zone_name.split("\n")[0], mar_change, mar_valid))
    may_stats.append((zone_name.split("\n")[0], may_change, may_valid))
    diff_mar_maps.append(diff_mar)
    diff_may_maps.append(diff_may)
    zone_extents.append(ext)

# --- Figure A: NDVI change maps (2 rows × 3 zones + shared colorbar) ---

fig = plt.figure(figsize=(16, 9))
gs = GridSpec(2, 4, figure=fig, width_ratios=[1, 1, 1, 0.04], hspace=0.25, wspace=0.15)

row_labels = [
    "Early Spring — March 2026",
    "Peak Spring Reference — May 2025",
]
zone_names_short = list(zones.keys())

for row, (maps, stats, label) in enumerate(
    zip(
        [diff_mar_maps, diff_may_maps],
        [mar_stats, may_stats],
        row_labels,
        strict=True,
    )
):
    for col in range(3):
        ax = fig.add_subplot(gs[row, col])
        im = ax.imshow(
            maps[col],
            cmap="RdYlGn",
            vmin=-0.4,
            vmax=0.4,
            extent=zone_extents[col],
            aspect="equal",
        )
        # Mean change badge
        ax.text(
            0.03,
            0.97,
            f"mean: {stats[col][1]:+.2f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray", alpha=0.85),
        )
        # Column headers (top row only)
        if row == 0:
            ax.set_title(zone_names_short[col], fontsize=12, fontweight="bold", pad=22)
        # Row subtitle on center column: sits between column headers and maps
        if col == 1:
            ax.text(
                0.5, 1.02, label,
                transform=ax.transAxes,
                ha="center", va="bottom",
                fontsize=11, fontweight="bold", color="#333",
                clip_on=False,
            )
        # Axis labels: only edges
        if col == 0:
            ax.set_ylabel("Northing (m)")
        else:
            ax.set_yticklabels([])
        if row == 1:
            ax.set_xlabel("Easting (m)")
        else:
            ax.set_xticklabels([])



# Shared colorbar
cbar_ax = fig.add_subplot(gs[:, 3])
cb = fig.colorbar(im, cax=cbar_ax)
cb.set_label("NDVI change", fontsize=11)
cb.ax.text(
    0.5, -0.03, "browning", transform=cb.ax.transAxes, ha="center", fontsize=8, color="#8B0000",
    clip_on=False,
)
cb.ax.text(
    0.5,
    1.03,
    "greening",
    transform=cb.ax.transAxes,
    ha="center",
    fontsize=8,
    color="#006400",
    clip_on=False,
)

fig.suptitle(
    "Elevation Gradient: NDVI Change vs November 2025 Baseline",
    fontsize=14,
    y=0.99,
)
plt.show()

# --- Zone stats summary (backs up Section 7 narrative) ---
print("\n=== Elevation Zone NDVI Change vs November baseline ===")
for (zm, mm, mv), (_, mp, pv) in zip(mar_stats, may_stats):
    print(f"  {zm:15s}  March ΔNDVI: {mm:+.2f}  May ΔNDVI: {mp:+.2f}  (valid: {mv:.0f}% / {pv:.0f}%)")
print("  (March: early spring | May: peak spring)")

In [ ]:
# --- Figure B: Summary bar chart + zone location map ---
fig, (ax_bars, ax_map) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2, 1]})

# Grouped bar chart: March vs May for each zone
zone_labels = [s[0] for s in mar_stats]
mar_vals = [s[1] for s in mar_stats]
may_vals = [s[1] for s in may_stats]
y = np.arange(len(zone_labels))
bar_h = 0.35

bars_mar = ax_bars.barh(
    y + bar_h / 2,
    mar_vals,
    bar_h,
    color="#e67e22",
    edgecolor="black",
    linewidth=0.5,
    label="March 2026 vs Nov",
)
bars_may = ax_bars.barh(
    y - bar_h / 2,
    may_vals,
    bar_h,
    color="#27ae60",
    edgecolor="black",
    linewidth=0.5,
    label="May 2025 vs Nov (reference)",
)
ax_bars.set_yticks(y)
ax_bars.set_yticklabels(zone_labels, fontsize=11)
ax_bars.set_xlabel("NDVI change from November baseline", fontsize=11)
ax_bars.axvline(x=0, color="gray", linewidth=0.5)
ax_bars.legend(loc="lower right", fontsize=10)

# Value annotations
for bars in [bars_mar, bars_may]:
    for bar in bars:
        w = bar.get_width()
        ax_bars.text(
            w + 0.008,
            bar.get_y() + bar.get_height() / 2,
            f"{w:+.2f}",
            va="center",
            fontsize=10,
            fontweight="bold",
        )

ax_bars.set_title("Phenological gradient: flat in March, steep by May", fontsize=12)

# Zone location mini-map
ax_map.imshow(rgb_720m, extent=ext_720, aspect="equal")
for zi, (_zn, coords) in enumerate(zones.items()):
    e0, e1 = coords["easting"]
    n0, n1 = coords["northing"]
    # Dashed outline signals "sample window", not the full zone
    rect = Rectangle(
        (e0, min(n0, n1)),
        e1 - e0,
        abs(n1 - n0),
        lw=1.8,
        edgecolor=zone_colors[zi],
        facecolor=zone_colors[zi],
        alpha=0.25,
        linestyle="--",
    )
    ax_map.add_patch(rect)
    # Label inside the box at top-left corner
    ax_map.text(
        e0 + 200,
        max(n0, n1) - 200,
        zone_labels[zi],
        fontsize=8,
        fontweight="bold",
        color="white",
        va="top",
        bbox=dict(boxstyle="round,pad=0.15", facecolor=zone_colors[zi], edgecolor="none", alpha=0.85),
    )
ax_map.set_title("Sampling windows (4 × 4 km each)", fontsize=10)
ax_map.set_xticks([])
ax_map.set_yticks([])

plt.tight_layout()
plt.show()

### What the gradient tells us

Each measurement comes from a 4 × 4 km sampling window within the broader Po Plain, Collio Hills, and Julian Alps elevation zones, which extend well beyond these windows across the full tile.

The top row (March vs November) shows **similar greening across all three zones** — Po Plain (ΔNDVI +0.18), Collio Hills (+0.17), Julian Alps (+0.19). The gradient hasn't developed yet in early March; all elevations are at the same early stage.

The bottom row (May vs November) reveals altitude-dependent phenology (the timing of leaf-out). By peak spring, the Po Plain has gained **+0.28 in NDVI** while the Julian Alps are at just **+0.10** — 2.8× less greening at altitude, and still behind the lowlands, which have already peaked. The 2.8× difference is consistent with Menzel et al. (2006)'s ~22-day delay per 1,000 m — our 3-date snapshot captures the magnitude of the gradient rather than deriving phenological timing directly.

The grouped bar chart makes it concrete: March bars are nearly flat across elevations, while May bars step down sharply from lowland to alpine. Adding a third date from the same archive cost only a few extra partial reads — no downloads, no format conversions. That’s what cloud-native access makes practical. Running this analysis in June with 2026 data (see Task 3 below) would capture the green wave climbing uphill in near real time.

#### Writing Results as Zarr — Completing the Cloud-Native Round-Trip

Cloud-native doesn't mean read-only. The same zarr library that opened our remote store can write the analysis output locally — preserving the structure and metadata so downstream tools can read it the same way.

In [ ]:
# # Write elevation-zone NDVI change maps to a local Zarr store
# # output_store = zarr.open("ndvi_elevation_analysis.zarr", mode="w")
# for (zm, _, _), diff_m, diff_p in zip(mar_stats, diff_mar_maps, diff_may_maps):
#     zone_key = zm.replace(" ", "_").lower()
#     grp = output_store.require_group(zone_key)
#     grp["ndvi_change_march"] = diff_m.astype("float32")
#     grp["ndvi_change_may"] = diff_p.astype("float32")
#     for var in ("ndvi_change_march", "ndvi_change_may"):
#         month = var.split("_")[2].capitalize()
#         grp[var].attrs.update({
#             "units": "1",
#             "valid_min": -1.0,
#             "valid_max": 1.0,
#             "long_name": f"NDVI change vs Nov 2025 baseline ({month})",
#         })

# output_store.attrs.update({
#     "Conventions": "CF-1.8",
#     "description": "NDVI change vs November 2025 baseline by elevation zone",
#     "crs": "EPSG:32633",
#     "sensor": "Sentinel-2 L2A",
#     "tile": "T33TVM",
# })
# print(f"Output store groups: {list(output_store.keys())}")
# for key in output_store.keys():
#     grp = output_store[key]
#     for var in grp.keys():
#         print(f"  {key}/{var}: {grp[var].shape}  attrs={dict(grp[var].attrs)}")

<hr>

## 8. The Bill: How Much Did We Actually Transfer?

Every read in this notebook was timed and logged. How much did cloud-native partial reads actually transfer?

Every `timed_read` call logged bytes and wall time. The waterfall chart shows per-read and cumulative data. These numbers show total decompressed bytes — the actual network transfer is smaller, so these are a conservative upper bound. The comparison bar shows what downloading the same bands at full tile resolution would cost: **2,351 MB**. Against that, 106 MB is a 22× reduction — the cloud-native advantage in one number.

The largest items are the full-tile SCL masks (three dates × 5490×5490 pixels). That's a deliberate choice: the elevation zones span the entire north-south extent of the tile, so we need cloud information across the full scene. Cropping the SCL to a single AOI would miss clouds hanging over the alpine zone 80 km north of the lowland zone.

In [ ]:
# --- Waterfall chart: cumulative data transfer ---
labels = [e["label"] for e in perf_log]
cumulative_mb = np.cumsum([e["bytes"] / 1e6 for e in perf_log])
individual_mb = [e["bytes"] / 1e6 for e in perf_log]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios": [2, 1]})

# Top: waterfall of cumulative transfer
colors = []
for e in perf_log:
    if "SCL" in e["label"]:
        colors.append("#e74c3c")  # red for quality data
    elif "720m" in e["label"] or "360m" in e["label"]:
        colors.append("#3498db")  # blue for overviews
    elif "elev" in e["label"]:
        colors.append("#9b59b6")  # purple for elevation zones
    else:
        colors.append("#2ecc71")  # green for analysis

ax1.bar(range(len(perf_log)), individual_mb, color=colors, edgecolor="white", linewidth=0.5)
ax1.plot(
    range(len(perf_log)), cumulative_mb, "k-o", markersize=3, linewidth=1.5, label="Cumulative"
)
ax1.set_xlabel("Read #")
ax1.set_ylabel("MB")
ax1.set_title("Data Transfer: Per-Read and Cumulative")
ax1.legend(loc="upper left")

# Bottom: comparison bar — honest baseline using only the bands we actually needed
total_mb = cumulative_mb[-1]
total_sec = sum(e["seconds"] for e in perf_log)
n_reads = len(perf_log)

# What downloading these bands at full tile would cost:
# Spring: B02-04,B08 @10m + B05,B8A,SCL @20m (4+3)
# Autumn: B04,B08 @10m + B05,B8A,SCL @20m (2+3)
# Peak:   B04,B08 @10m + SCL @20m (2+1)
px_10m = 10980 * 10980  # pixels per 10m band
px_20m = 5490 * 5490  # pixels per 20m band
bytes_per_px = 2  # uint16 (raw Sentinel-2 encoding)
bands_10m_total = 4 + 2 + 2  # 8 full-tile 10m bands across 3 scenes
bands_20m_total = 3 + 3 + 1  # 7 full-tile 20m bands across 3 scenes
full_bands_mb = (bands_10m_total * px_10m + bands_20m_total * px_20m) * bytes_per_px / 1e6

categories = ["This notebook\n(partial reads)", "Full download\n(same bands, full tile)"]
values = [total_mb, full_bands_mb]
bar_colors = ["#2ecc71", "#e74c3c"]
bars = ax2.barh(categories, values, color=bar_colors, edgecolor="black", linewidth=0.5, height=0.5)
ax2.set_xlabel("MB")
ax2.set_title("Cloud-Native Partial Reads vs Full Tile Download")
for bar, val in zip(bars, values, strict=False):
    ax2.text(
        bar.get_width() + 20,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.0f} MB",
        va="center",
        fontsize=12,
        fontweight="bold",
    )

fig.suptitle("The Data Bill: How Much Did We Actually Transfer?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

reduction = full_bands_mb / total_mb
print("=== Performance Summary ===")
print(f"Reads:       {n_reads}")
print(f"Bytes read:  {total_mb:.1f} MB  (decompressed; actual wire transfer is smaller)")
print(f"Wall time:   {total_sec:.1f} s")
print(f"Full tile equivalent (same bands only): {full_bands_mb:.0f} MB")
print(f"Reduction:   {reduction:.0f}x")

# --- Grouped read summary ---
groups = {
    "Overviews (720m, 360m)": [],
    "SCL masks (full tile)": [],
    "AOI bands (10m, 20m)": [],
    "Elevation zones": [],
}
for e in perf_log:
    if "720m" in e["label"] or "360m" in e["label"]:
        groups["Overviews (720m, 360m)"].append(e)
    elif "SCL" in e["label"]:
        groups["SCL masks (full tile)"].append(e)
    elif "elev" in e["label"]:
        groups["Elevation zones"].append(e)
    else:
        groups["AOI bands (10m, 20m)"].append(e)

print(f"\n{'Category':<25s} {'Reads':>5s} {'MB':>8s} {'Sec':>6s}")
print(f"{'---':<25s} {'---':>5s} {'---':>8s} {'---':>6s}")
for cat, entries in groups.items():
    n = len(entries)
    mb = sum(e["bytes"] / 1e6 for e in entries)
    sec = sum(e["seconds"] for e in entries)
    print(f"{cat:<25s} {n:5d} {mb:8.1f} {sec:6.1f}")

## 9. Why Format Consistency Matters

This analysis relied on format consistency at every step — coordinate metadata for pixel lookups, the zarr.json table of contents for discovery, sharding for efficient reads. Cloud-native Earth observation depends on agreement between three layers: the format on disk, the metadata catalog, and the reader software. When any link breaks — a catalog listing the wrong version, a reader expecting a different chunk layout, a viewer ignoring coordinate metadata — the whole chain fails.

The media type mismatch we caught in Section 2 is a real-world example of what happens when catalog and data drift apart. The [GeoZarr conformance test suite](https://github.com/zarr-developers/geozarr-spec/pull/127) helps surface these gaps systematically, and [upstream fixes](https://github.com/EOPF-Explorer/data-pipeline/pull/94) close them — but always probe the store rather than trusting catalog metadata alone.

Consistency also extends across the constellation: the EOPF sample service has a Sentinel-2C acquisition of this tile (2026-03-08, 1.1% cloud) available in identical format. Swapping the item ID in Section 2 is the only change needed — the rest of this notebook runs unchanged. That's format stability in practice.

<hr>

## 💪 Now it is your turn

### Task 1: Compute a Water Index

*A quick win — one function call and a plot, using tools you've already seen.*

Use the same `compute_index` helper to compute NDWI (Normalized Difference Water Index) for the spring scene. NDWI uses green (B03) and NIR (B08) — water absorbs NIR and reflects green, the opposite of vegetation:

```python
ndwi = compute_index(spring, "b03", "b08", AOI_10m, "r10m", "NDWI")
plt.imshow(ndwi, cmap="RdBu", vmin=-0.5, vmax=0.5,
           extent=geo_extent_aoi(TRANSFORM, AOI_10m))
plt.colorbar(label="NDWI")
plt.title("NDWI — positive = water, negative = land")
plt.show()
```

Hypothesis: fewer than 5% of AOI pixels will show NDWI > 0, concentrated along river channels and irrigation features. Count them and describe what you see.

### Task 2: Compare Resolutions

*Explore the tradeoff between spatial detail and data volume.*

Compute NDVI at **20m** (`r20m` group) for the same AOI and compare with the 10m result:

```python
ndvi_20m = compute_index(spring, "b08", "b04", AOI_20m, "r20m", "NDVI-20m")
print(f"10m mean NDVI: {np.nanmean(spring_ndvi):.3f}  shape: {spring_ndvi.shape}")
print(f"20m mean NDVI: {np.nanmean(ndvi_20m):.3f}  shape: {ndvi_20m.shape}")
```

Hypothesis: mean NDVI values at 10m and 20m should agree within 0.02, but standard deviation will be higher at 10m due to unresolved field boundaries. Test this by computing both and comparing. At what pixel scale does the difference become negligible?

### Task 3: Track the Green Wave Uphill

*The most ambitious task — add a fourth date and test a hypothesis.*

Query the STAC API for a **July 2025** scene and add it to the elevation gradient analysis:

```python
summer_results = client.search(
    collections=["sentinel-2-l2a"], bbox=BBOX,
    datetime="2025-07-01/2025-07-31",
    query={"eo:cloud_cover": {"lt": 10}}, max_items=20,
)
summer_item = next(item for item in summer_results.items() if TILE in item.id)
summer = open_scene(summer_item)
```

Hypothesis: by July, snowmelt and accumulated warmth should push green-up into the Julian Alps. The NDVI change at 1,500 m should be strongly positive, while the lowlands — already green since March — may show stable or declining NDVI as summer stress sets in. Does the data confirm this?

## Conclusion

We set out to detect spring green-up across a 2,000 m elevation gradient, transferring around 100 MB from the EOPF Zarr Sample Service. Here's what made it work:

- **Progressive zoom** (Section 4): three resolution levels in one store let us go from country overview to vineyard rows, reading only what we needed at each scale
- **One request for the full picture** (Section 3): a single table-of-contents file replaced hundreds of individual metadata lookups, and sharding let us read small areas without fetching entire bands
- **Everything in one place** (Section 5): cloud masks and spectral bands at multiple resolutions all live in the same store — no grid alignment code, no separate downloads

The result: a complete three-date vegetation change analysis — autumn baseline, early spring, and peak spring — reading **106 MB** against a 2,351 MB full-tile equivalent: a 22× reduction. That ratio scales: monitor the same tile weekly for five years and you transfer ~9 GB instead of ~200 GB. The altitude-dependent pattern visible in the May comparison — lowlands peaked while alpine meadows are just emerging — shows that reading just the right pixels from the right place makes spatially targeted science practical, even over slow connections.

## What's next?

This notebook analyzed three dates. The natural next step is a full time series — stacking monthly scenes to track the green wave as it climbs from lowland to alpine across an entire growing season. Task 3 above is a first step in that direction. The [EOPF 101 book](https://eopf-toolkit.github.io/eopf-101/) covers multi-temporal analysis workflows in depth.

For higher-level workflows, the [eopf-toolkit](https://eopf-toolkit.github.io/eopf-toolkit/) wraps these same stores into familiar xarray-style interfaces — this notebook shows what's happening under the hood. To browse EOPF stores interactively without writing code, the [GeoZarr plugin for QGIS](https://github.com/wietzesuijker/qgis-geozarr) reads the same data with progressive zoom built into the map canvas.